# 03 — Patch-Text Similarity Localisation

Core experimental pipeline. For each of the four prompt conditions and each annotated image,
computes patch-level cosine similarity between BiomedCLIP patch tokens and text embeddings,
producing 14×14 localisation heatmaps. Evaluates against CheXlocalize radiologist GT masks.

**Method:**
1. Encode text prompt → 768-dim text embedding (BiomedCLIP text encoder)
2. Encode image → 196 patch tokens of shape (196, 768) (BiomedCLIP vision encoder)
3. Cosine similarity: each patch token vs text embedding → 14×14 heatmap
4. Upsample to original image resolution
5. Threshold at 50th percentile → binary localisation mask
6. Compute IoU and hit rate vs CheXlocalize GT mask
7. Compute alignment ratio: mean similarity of pathology patches / background patches

**Requires:**
- `$DRIVE_BASE/prompts/all_prompts.json` from notebook 02
- `$DRIVE_BASE/chexpert_val/` — CheXpert validation set
- `$DRIVE_BASE/chexlocalize/chexlocalize/CheXlocalize/gt_segmentations_val.json`

**Output:** `$DRIVE_BASE/results/localisation_results.json` + figures

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install open_clip_torch datasets pycocotools scipy matplotlib -q

import os, json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import open_clip
from PIL import Image
from scipy.ndimage import zoom
from pycocotools import mask as mask_utils
from collections import defaultdict
from datasets import load_from_disk

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
DRIVE_BASE  = '/content/drive/MyDrive/medvit-graphrag'
RESULTS_DIR = f'{DRIVE_BASE}/results'
FIGURES_DIR = f'{DRIVE_BASE}/figures'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f'Device: {DEVICE}')

In [ ]:
# load data and prompts
ds = load_from_disk(f'{DRIVE_BASE}/chexpert_val')
print(f'CheXpert validation: {len(ds)} images')

CHEXLOC_FILE = f'{DRIVE_BASE}/chexlocalize/chexlocalize/CheXlocalize/gt_segmentations_val.json'
with open(CHEXLOC_FILE) as f:
    chexloc_data = json.load(f)
print(f'CheXlocalize GT masks: {len(chexloc_data)} annotated images')

with open(f'{DRIVE_BASE}/prompts/all_prompts.json') as f:
    ALL_PROMPTS = json.load(f)
print(f'Prompt conditions: {list(ALL_PROMPTS.keys())}')

PATHOLOGY_MAP = {
    'Pleural Effusion': 'Pleural Effusion',
    'Cardiomegaly':     'Cardiomegaly',
    'Edema':            'Edema',
    'Lung Opacity':     'Airspace Opacity',
}

def get_label(row: dict) -> str:
    if row['No Finding'] == 3:
        return 'Normal'
    for col in ['Pleural Effusion', 'Cardiomegaly', 'Edema', 'Lung Opacity']:
        if row[col] == 3:
            return col
    return 'Other'

all_labels = [get_label(ds[i]) for i in range(len(ds))]
label_arr  = np.array(all_labels)

def chexpert_path_to_chexloc_key(path: str) -> str:
    return path.replace('CheXpert-v1.0-small/valid/', '').replace('.jpg', '').replace('/', '_')

In [ ]:
# load BiomedCLIP
MODEL_ID = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
print(f'Loading {MODEL_ID} ...')

clip_model, _, img_processor = open_clip.create_model_and_transforms(MODEL_ID)
tokenizer = open_clip.get_tokenizer(MODEL_ID)
clip_model = clip_model.to(DEVICE)
clip_model.eval()

vit      = clip_model.visual.trunk
n_layers = len(vit.blocks)
GRID     = 14  # 224 / 16 = 14 patches per side
print(f'Loaded. Patch grid: {GRID}x{GRID} = {GRID**2} patches')

In [ ]:
# patch token extraction via forward hook on final transformer block
patch_tokens_cache: dict = {}

def _patch_hook(module, input, output):
    # output shape: (batch, seq_len, hidden) where seq_len = 197 (CLS + 196 patches)
    patch_tokens_cache['tokens'] = output[:, 1:, :].detach()  # drop CLS

hook = vit.blocks[-1].register_forward_hook(_patch_hook)

def extract_patch_tokens(img: Image.Image) -> torch.Tensor:
    """
    Returns patch tokens of shape (196, 768) for a single image.
    L2-normalised for cosine similarity computation.
    """
    x = img_processor(img.convert('RGB')).unsqueeze(0).to(DEVICE)
    patch_tokens_cache.clear()
    with torch.no_grad():
        _ = vit(x)
    tokens = patch_tokens_cache['tokens'].squeeze(0)  # (196, 768)
    return F.normalize(tokens, dim=-1)


def encode_text(prompt: str) -> torch.Tensor:
    """
    Returns L2-normalised text embedding of shape (768,).
    """
    tokens = tokenizer([prompt]).to(DEVICE)
    with torch.no_grad():
        text_emb = clip_model.encode_text(tokens)
    return F.normalize(text_emb, dim=-1).squeeze(0)  # (768,)


def patch_text_similarity_map(img: Image.Image, prompt: str) -> np.ndarray:
    """
    Returns 14×14 cosine similarity map between each patch token and the text embedding.
    Values normalised to [0, 1].
    """
    patch_tokens = extract_patch_tokens(img)   # (196, 768)
    text_emb     = encode_text(prompt)          # (768,)
    similarities = (patch_tokens @ text_emb).cpu().numpy()  # (196,)
    sim_map      = similarities.reshape(GRID, GRID)         # (14, 14)
    sim_map      = (sim_map - sim_map.min()) / (sim_map.max() - sim_map.min() + 1e-10)
    return sim_map


print('Patch token extraction pipeline ready.')

In [ ]:
# evaluation utilities

def decode_rle_mask(rle: dict) -> np.ndarray:
    return mask_utils.decode(rle).astype(np.float32)


def upsample_map(sim_map: np.ndarray, H: int, W: int) -> np.ndarray:
    return zoom(sim_map, (H / sim_map.shape[0], W / sim_map.shape[1]), order=1)


def compute_iou(sim_up: np.ndarray, gt_mask: np.ndarray) -> float:
    pred  = (sim_up >= np.percentile(sim_up, 50)).astype(np.float32)
    inter = (pred * gt_mask).sum()
    union = ((pred + gt_mask) >= 1).sum()
    return float(inter / (union + 1e-10))


def compute_hit(sim_up: np.ndarray, gt_mask: np.ndarray) -> float:
    py, px = np.unravel_index(sim_up.argmax(), sim_up.shape)
    return float(gt_mask[py, px] > 0)


def compute_alignment_ratio(
    sim_map_14: np.ndarray,
    gt_mask: np.ndarray,
    patch_size: int = 16,
    img_w: int = 224,
    img_h: int = 224
) -> float:
    """
    Alignment ratio = mean similarity of pathology patches / mean similarity of background patches.
    Patches are labelled pathology if their 16x16 region has >50% overlap with the GT mask
    resized to 224x224.
    Ratio > 1 means text discriminates pathology from background.
    """
    gt_224 = zoom(gt_mask, (img_h / gt_mask.shape[0], img_w / gt_mask.shape[1]), order=0)
    pathology_sims, background_sims = [], []
    for r in range(GRID):
        for c in range(GRID):
            patch_mask = gt_224[r*patch_size:(r+1)*patch_size, c*patch_size:(c+1)*patch_size]
            if patch_mask.mean() > 0.5:
                pathology_sims.append(sim_map_14[r, c])
            else:
                background_sims.append(sim_map_14[r, c])
    if not pathology_sims or not background_sims:
        return float('nan')
    return float(np.mean(pathology_sims) / (np.mean(background_sims) + 1e-10))


print('Evaluation utilities ready.')

In [ ]:
# main evaluation loop
CONDITIONS = ['flat', 'expanded', 'graphrag', 'graphrag_spatial']
results    = defaultdict(list)  # condition -> list of per-image result dicts

print(f'Running localisation evaluation on {len(ds)} images x {len(CONDITIONS)} conditions...\n')

for i in range(len(ds)):
    label = all_labels[i]
    if label not in PATHOLOGY_MAP:
        continue

    key = chexpert_path_to_chexloc_key(ds[i]['Path'])
    if key not in chexloc_data:
        continue

    chexloc_pathology = PATHOLOGY_MAP[label]
    entry = chexloc_data[key]
    if chexloc_pathology not in entry or entry[chexloc_pathology] is None:
        continue

    gt_mask = decode_rle_mask(entry[chexloc_pathology])
    H, W    = gt_mask.shape
    img     = ds[i]['image'].convert('RGB')

    for condition in CONDITIONS:
        prompt  = ALL_PROMPTS[condition][label]
        sim_map = patch_text_similarity_map(img, prompt)   # (14, 14)
        sim_up  = upsample_map(sim_map, H, W)

        results[condition].append({
            'idx':             i,
            'label':           label,
            'iou':             compute_iou(sim_up, gt_mask),
            'hit':             compute_hit(sim_up, gt_mask),
            'alignment_ratio': compute_alignment_ratio(sim_map, gt_mask),
            'peak_sim':        float(sim_map.max()),
            'mean_sim':        float(sim_map.mean()),
        })

    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(ds)} images processed')

hook.remove()

out_path = f'{RESULTS_DIR}/localisation_results.json'
with open(out_path, 'w') as f:
    json.dump(dict(results), f, indent=2)
print(f'\nResults saved to {out_path}')
print(f'Matched images per condition: {len(results["flat"])}')

In [ ]:
# results summary table
print('Localisation performance by condition and pathology:')
print(f'{"Condition":<20} {"Pathology":<20} {"n":>4} {"IoU":>8} {"Hit":>8} {"Align":>8}')
print('-' * 72)

summary = {}
for condition in CONDITIONS:
    by_label = defaultdict(list)
    for r in results[condition]:
        by_label[r['label']].append(r)

    all_ious, all_hits, all_align = [], [], []
    summary[condition] = {}

    for label in sorted(PATHOLOGY_MAP.keys()):
        items = by_label[label]
        if not items:
            continue
        ious   = [r['iou'] for r in items]
        hits   = [r['hit'] for r in items]
        aligns = [r['alignment_ratio'] for r in items if not np.isnan(r['alignment_ratio'])]
        all_ious.extend(ious)
        all_hits.extend(hits)
        all_align.extend(aligns)
        summary[condition][label] = {
            'n': len(items), 'iou': np.mean(ious),
            'hit': np.mean(hits), 'alignment_ratio': np.mean(aligns) if aligns else float('nan')
        }
        print(f'{condition:<20} {label:<20} {len(items):>4} '
              f'{np.mean(ious):>8.4f} {np.mean(hits):>8.4f} '
              f'{np.mean(aligns) if aligns else float("nan"):>8.4f}')

    summary[condition]['Overall'] = {
        'n': len(all_ious), 'iou': np.mean(all_ious),
        'hit': np.mean(all_hits), 'alignment_ratio': np.mean(all_align)
    }
    print(f'{condition:<20} {"Overall":<20} {len(all_ious):>4} '
          f'{np.mean(all_ious):>8.4f} {np.mean(all_hits):>8.4f} '
          f'{np.mean(all_align):>8.4f}')
    print()

with open(f'{RESULTS_DIR}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

In [ ]:
# figure 1: IoU comparison across conditions per pathology
CONDITION_LABELS = {
    'flat':             'A: Flat',
    'expanded':         'B: Expanded',
    'graphrag':         'C: GraphRAG',
    'graphrag_spatial': 'D: GraphRAG\n(spatial)',
}
PATHOLOGY_COLORS = {
    'Pleural Effusion': '#3498db',
    'Cardiomegaly':     '#e74c3c',
    'Edema':            '#9b59b6',
    'Lung Opacity':     '#f39c12',
}

fig, axes = plt.subplots(1, len(PATHOLOGY_MAP), figsize=(16, 5), sharey=True)
x         = np.arange(len(CONDITIONS))

for ax, label in zip(axes, sorted(PATHOLOGY_MAP.keys())):
    ious = [summary[c][label]['iou'] for c in CONDITIONS]
    bars = ax.bar(x, ious, color=PATHOLOGY_COLORS[label], alpha=0.8, width=0.6)
    for bar, val in zip(bars, ious):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_title(label, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([CONDITION_LABELS[c] for c in CONDITIONS], fontsize=7)
    ax.set_ylim(0, max(ious) * 1.3)
    ax.grid(alpha=0.3, axis='y')
    if ax == axes[0]:
        ax.set_ylabel('IoU vs CheXlocalize GT')

plt.suptitle('Zero-Shot Localisation IoU: Flat vs GraphRAG Prompting (BiomedCLIP)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/iou_by_condition.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# figure 2: alignment ratio across conditions
fig, axes = plt.subplots(1, len(PATHOLOGY_MAP), figsize=(16, 5), sharey=True)

for ax, label in zip(axes, sorted(PATHOLOGY_MAP.keys())):
    ratios = [summary[c][label]['alignment_ratio'] for c in CONDITIONS]
    bars   = ax.bar(x, ratios, color=PATHOLOGY_COLORS[label], alpha=0.6, width=0.6)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=0.8, label='No discrimination (1.0)')
    for bar, val in zip(bars, ratios):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_title(label, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([CONDITION_LABELS[c] for c in CONDITIONS], fontsize=7)
    ax.grid(alpha=0.3, axis='y')
    if ax == axes[0]:
        ax.set_ylabel('Alignment Ratio (pathology / background)')
        ax.legend(fontsize=7)

plt.suptitle('Alignment Ratio: Pathology vs Background Patch Similarity', fontsize=11)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/alignment_ratio_by_condition.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# figure 3: qualitative visualisation
# for each pathology, show one image with all 4 condition heatmaps side by side

# reload hook for visualisation
hook = vit.blocks[-1].register_forward_hook(_patch_hook)

for label in sorted(PATHOLOGY_MAP.keys()):
    # find first matched image for this pathology
    matched = [r for r in results['flat'] if r['label'] == label]
    if not matched:
        continue
    idx = matched[0]['idx']
    img = ds[idx]['image'].convert('RGB')

    key               = chexpert_path_to_chexloc_key(ds[idx]['Path'])
    chexloc_pathology = PATHOLOGY_MAP[label]
    gt_mask           = decode_rle_mask(chexloc_data[key][chexloc_pathology])
    H, W              = gt_mask.shape
    img_display       = np.array(img.resize((W, H)))

    fig, axes = plt.subplots(1, len(CONDITIONS) + 2, figsize=(20, 4))

    # original image
    axes[0].imshow(img_display, cmap='gray')
    axes[0].set_title('Original', fontsize=8)
    axes[0].axis('off')

    # GT mask
    axes[1].imshow(img_display, cmap='gray')
    axes[1].imshow(gt_mask, cmap='cool', alpha=0.4)
    axes[1].set_title('GT mask', fontsize=8)
    axes[1].axis('off')

    # per-condition heatmaps
    for ax, condition in zip(axes[2:], CONDITIONS):
        prompt  = ALL_PROMPTS[condition][label]
        sim_map = patch_text_similarity_map(img, prompt)
        sim_up  = upsample_map(sim_map, H, W)
        iou_val = compute_iou(sim_up, gt_mask)
        ax.imshow(img_display, cmap='gray')
        ax.imshow(sim_up, cmap='hot', alpha=0.5, interpolation='bilinear')
        ax.set_title(f'{CONDITION_LABELS[condition]}\nIoU={iou_val:.3f}', fontsize=7)
        ax.axis('off')

    plt.suptitle(f'{label}', fontsize=10)
    plt.tight_layout()
    fname = label.replace(' ', '_').lower()
    plt.savefig(f'{FIGURES_DIR}/qualitative_{fname}.png', dpi=150, bbox_inches='tight')
    plt.show()

hook.remove()

In [ ]:
# failure decomposition
# for images where graphrag_spatial fails to improve over flat,
# classify as encoder-side vs graph-side failure

print('Failure decomposition: encoder-side vs graph-side\n')
print(f'{"Pathology":<20} {"Total failures":>15} {"Encoder-side":>14} {"Graph-side":>12}')
print('-' * 65)

flat_by_idx    = {r['idx']: r for r in results['flat']}
spatial_by_idx = {r['idx']: r for r in results['graphrag_spatial']}

for label in sorted(PATHOLOGY_MAP.keys()):
    encoder_failures = 0
    graph_failures   = 0
    total_failures   = 0

    for idx in flat_by_idx:
        if flat_by_idx[idx]['label'] != label:
            continue
        flat_r    = flat_by_idx[idx]
        spatial_r = spatial_by_idx.get(idx)
        if spatial_r is None:
            continue

        # failure: graphrag_spatial does not improve over flat
        if spatial_r['iou'] <= flat_r['iou']:
            total_failures += 1
            flat_align = flat_r['alignment_ratio']
            if np.isnan(flat_align):
                continue
            # encoder-side: flat alignment ratio already near 1 (no spatial structure in tokens)
            # graph-side: flat alignment ratio > 1 (tokens have spatial structure but wrong location)
            if flat_align < 1.05:
                encoder_failures += 1
            else:
                graph_failures += 1

    print(f'{label:<20} {total_failures:>15} {encoder_failures:>14} {graph_failures:>12}')

print('\nEncoder-side: flat alignment ratio < 1.05 — ViT lacks spatial clinical structure for this image.')
print('Graph-side: flat alignment ratio >= 1.05 — spatial structure exists but prompt targets wrong region.')